In [ ]:
from pystac.client import Client
from odc.stac import load
import xarray as xr
from utils import mask_land, mask_deeps, make_indices, do_prediction, locations
import joblib
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor

from ipyleaflet import basemaps
import folium

import warnings
warnings.filterwarnings("ignore")

In [2]:
catalog = Client.open("https://earth-search.aws.element84.com/v1")
collection = "sentinel-2-l2a"

In [3]:
location = locations.suva
daterange = "2024"

In [4]:
# Consider removing very cloudy scenes...
items = catalog.search(
    collections=[collection],
    bbox=location.bbox,
    datetime=daterange,
    query={"eo:cloud_cover": {"lt": 50}},
).item_collection()

print(f"Found {len(items)} items")

Found 63 items


In [5]:
data = load(
    items,
    bbox=location.bbox,
    epsg="utm",
    measurements=[
        "scl",
        "nir",
        "red",
        "blue",
        "green",
        "nir08",
        "nir09",
        "swir16",
        "swir22",
        "coastal",
        "rededge1",
        "rededge2",
        "rededge3",
    ],
    chunks={"x": 2048, "y": 2048},
    nodata=0,
    groupby="solar_day",
)

# Mask clouds
mask = data.scl.isin([3, 8, 9, 10])
data = data.where(~mask)
data = make_indices(data)

# Mask land
data = mask_land(data)

# # Mask deep water
data = mask_deeps(data)

data = data.drop_vars("scl")

data

<xarray.Dataset> Size: 15GB
Dimensions:      (time: 39, y: 2232, x: 2135)
Coordinates:
  * y            (y) float64 18kB 8.009e+06 8.009e+06 ... 7.987e+06 7.987e+06
  * x            (x) float64 17kB 6.481e+05 6.481e+05 ... 6.694e+05 6.694e+05
    spatial_ref  int32 4B 32760
  * time         (time) datetime64[ns] 312B 2024-01-03T22:20:59.099000 ... 20...
Data variables: (12/20)
    nir          (time, y, x) float32 743MB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    red          (time, y, x) float32 743MB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    blue         (time, y, x) float32 743MB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    green        (time, y, x) float32 743MB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    nir08        (time, y, x) float32 743MB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    nir09        (time, y, x) float32 743MB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    ...           ...
    mndwi        (time, y, x) float32 743MB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    ndti         (time, y, x) float32 743MB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    stumpf       (time, y, x) float32 743MB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    bg           (time, y, x) float32 743MB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    br           (time, y, x) float32 743MB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    ln_bg        (time, y, x) float32 743MB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>

In [6]:
# Preview
# data.isel(time=0).odc.explore(bands=["red", "green", "blue"], vmin=0, vmax=2000)

# # Pre-load data
# data = data.compute()

In [ ]:
model = joblib.load("models/2025_03_10_randomforest_both_masks.joblib")

def predict_for_day(day):
    return do_prediction(data.sel(time=day), model).compute()

with ThreadPoolExecutor() as executor:
    predictions_list = list(tqdm(executor.map(predict_for_day, data.time), total=len(data.time)))

# Concatenate them all together again
predictions = xr.concat(predictions_list, dim="time").to_dataset(name="elevation")

predictions

100%|██████████| 39/39 [07:06<00:00, 10.93s/it]  


<xarray.Dataset> Size: 1GB
Dimensions:      (y: 2232, x: 2135, time: 39)
Coordinates:
  * y            (y) float64 18kB 8.009e+06 8.009e+06 ... 7.987e+06 7.987e+06
  * x            (x) float64 17kB 6.481e+05 6.481e+05 ... 6.694e+05 6.694e+05
    spatial_ref  int32 4B 32760
  * time         (time) datetime64[ns] 312B 2024-01-03T22:20:59.099000 ... 20...
Data variables:
    elevation    (time, y, x) float64 1GB nan nan nan nan ... nan nan nan nan

In [ ]:
# Plot all the timesteps
predictions.elevation.plot.imshow(col="time", col_wrap=2, cmap="Blues_r", robust=True, size=6)

In [ ]:
# Clean up the data by removing pixels that only had predictions sometimes
count = predictions.elevation.count(dim="time")
total = len(predictions.time)

mask = count > (total * 0.25)  # At least 10% of the time there was a prediction

mean = predictions.elevation.mean(dim="time")
mean = mean.where(mask)

# Make a fancy map
centroid = list(predictions.odc.geobox.geographic_extent.centroid.coords[0])[::-1]
m = folium.Map(location=centroid, zoom_start=12)
_ = folium.TileLayer(tiles=basemaps.Esri.WorldImagery).add_to(m)

count.odc.add_to(m, cmap="Reds", name="Count")
mean.odc.add_to(m, cmap="Blues_r", name="Depth")

folium.LayerControl().add_to(m)

m

In [ ]:
mean.odc.write_cog("tuvalu_clean.tif", overwrite=True)